In [ ]:
import os
import requests

base_url = 'https://zsa.zamstats.gov.zm:8443/cpi/'
user = 'testapi'
password = 'ZambiaZambia1'

if not all([base_url, user, password]):
    raise ValueError("Please set SS_BASE_URL, SS_USER, and SS_PASSWORD")

resp = requests.get(
    f"{base_url}/api/v1/questionnaires",
    auth=(user, password),
    timeout=30,
)

print(resp.status_code)

In [ ]:
session = requests.Session()
session.auth = (user, password)


# Both calls use the same credentials automatically
questionnaires = session.get(f"{base_url}/api/v1/questionnaires", timeout=30)
interviewers = session.get(f"{base_url}/api/v1/supervisors", timeout=30)

In [ ]:
questionnaires.json()

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
QUESTIONNAIRE_NAME = "Server test questionnaire"

resp = session.get(
    f"{base_url}/api/v1/questionnaires",
    timeout=30,
)
resp.raise_for_status()

questionnaires = resp.json()["Questionnaires"]

# Find all versions that match the name
matches = [
    q for q in questionnaires
    if q["Title"] == QUESTIONNAIRE_NAME
]

if not matches:
    raise ValueError(f"No questionnaire found with title: {QUESTIONNAIRE_NAME}")

# Pick the latest version
latest = max(matches, key=lambda q: q["Version"])

q_id = latest["QuestionnaireId"]
q_version = latest["Version"]
q_variable = latest["Variable"]

logger.info("Found: %s (id=%s, version=%s)", latest["Title"], q_id, q_version)

In [ ]:
export_type = "Tabular"  # or "Stata", "SPSS", "Binary", "Paradata" etc.
interview_status = "All"  # or "Completed" etc.

resp = session.post(
    f"{base_url}/api/v2/export",
    json={
        "QuestionnaireId": f"{q_id}${q_version}",
        "ExportType": export_type,
        "InterviewStatus": interview_status,
    },
    timeout=30,
)
resp.raise_for_status()
resp.json()


In [ ]:

job = resp.json()
job_id = job["JobId"]

logger.info(f"Export job started: {job_id}")

In [ ]:
import time

while True:
    resp = session.get(
        f"{base_url}/api/v2/export/{job_id}",
        timeout=30,
    )
    resp.raise_for_status()

    status = resp.json()["ExportStatus"]
    logger.info("Export status: %s", status)

    if status == "Completed":
        break

    if status == "Fail":
        raise RuntimeError("Export failed on the server")

    time.sleep(10)  # wait 10 seconds before checking again

In [ ]:
resp.json()

In [ ]:
from pathlib import Path
import datetime

resp = session.get(
    f"{base_url}/api/v2/export/{job_id}/file",
    timeout=120,
)
resp.raise_for_status()

timestamp = datetime.datetime.now(datetime.UTC).strftime("%Y%m%dT%H%M%S")
file_name = f"{q_variable}_v{q_version}_{export_type}_{timestamp}.zip"

output_path = Path("exports") / file_name
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_bytes(resp.content)

logger.info("Saved export to %s (%d bytes)", output_path, len(resp.content))